In [1]:
import requests
import time
import pandas as pd
import os

In [2]:
def filter_european(score):
    is_european = False
    temp = score.get("samples_training", [])
    if len(temp) > 0:
        ancestry = temp[0].get("ancestry_broad", [])
        if ancestry == "European":
            is_european = True
        else:
            pass
    if not is_european:
        temp = score.get("samples_variants", [])
        if len(temp) > 0:
            ancestry = temp[0].get("ancestry_broad", [])
            if ancestry == "European":
                is_european = True
            else:
                pass
    return is_european

def get_url(score):
    temp = score.get("ftp_harmonized_scoring_files", [])
    if len(temp) > 0:
        url = temp.get("GRCh38", []).get("positions", [])
        return url
    return False

def get_associdated_pgs_ids(score):
    temp = score.get("associated_pgs_ids", [])
    if len(temp) > 0:
        return temp
    return False

In [8]:
import os

output_path = os.path.join(os.getcwd(), "pgs_id_list_260410.csv")

if os.path.exists(output_path):
    icd2dsp = pd.read_csv(output_path, index_col=None)
    print(f"Loaded existing file with {len(icd2dsp)} rows")
    print(f"Rows missing pgs_ids: {icd2dsp['pgs_ids'].isna().sum()}")
else:
    icd2dsp = pd.read_csv(os.path.join(os.getcwd(), "trait_list_260410.csv"), index_col=None)
    icd2dsp["icd_root"] = icd2dsp["icd"].astype(str).str[:3]
    cols = list(icd2dsp.columns)
    cols.remove("icd_root")
    cols.insert(3, "icd_root")
    icd2dsp = icd2dsp[cols]
    icd2dsp["pgs_ids"] = None
    icd2dsp["pgs_api_num"] = None
    icd2dsp["pgs_urls"] = None
    print(f"Created new dataframe with {len(icd2dsp)} rows")

base_url = "https://www.pgscatalog.org/rest/trait/search"

for idx, row in icd2dsp.iterrows():
    # skip rows that already have pgs_ids
    if pd.notna(row["pgs_ids"]):
        continue

    time.sleep(5)
    icd = row["icd"]
    description = row["ontology"]

    params = {"term": description}
    response = requests.get(base_url, params=params)

    if response.status_code != 200:
        print(f"Failed for {icd}: {description}")
        continue

    data = response.json()

    pgs_ids = []
    while True:
        for s in data["results"]:
            temp = get_associdated_pgs_ids(s)
            if temp:
                pgs_ids.extend(temp)

        next_url = data.get("next")
        if not next_url:
            break

        r = requests.get(next_url)
        if r.status_code != 200:
            print(f"Pagination failed for {icd}")
            break
        data = r.json()

    pgs_ids = list(set(pgs_ids))
    icd2dsp.at[idx, "pgs_ids"] = str(pgs_ids)
    icd2dsp.at[idx, "pgs_api_num"] = int(len(pgs_ids))

    pgs_urls = []
    for pgs_id in pgs_ids:
        temp_url_api = "https://www.pgscatalog.org/rest/score/" + pgs_id
        r = requests.get(temp_url_api)
        data_score = r.json()
        temp_url = get_url(data_score)
        pgs_urls.extend([temp_url])
        time.sleep(0.2)

    icd2dsp.at[idx, "pgs_urls"] = str(pgs_urls)
    print(f"Finish {icd} | pgs: {len(pgs_ids)}, urls: {len(pgs_urls)}")

    # save after each row in case of interruption
    icd2dsp.to_csv(output_path, index=False)
    time.sleep(0.2)

print("Done.")

Loaded existing file with 39 rows
Rows missing pgs_ids: 4
Finish E119 | pgs: 209, urls: 209
Finish K5190 | pgs: 5, urls: 5
Finish F0150 | pgs: 1, urls: 1
Finish C880 | pgs: 1, urls: 1
Done.


In [5]:
icd2dsp.to_csv(os.path.join(os.getcwd(),"pgs_id_list_260410.csv"), index=False)

In [ ]:
import requests
import time

url = "https://www.pgscatalog.org/rest/score/all"

euro_urls = []

while url:
    r = requests.get(url)
    data = r.json()
    
    euro_data = [s for s in data["results"] if filter_european(s)]
    euro_url = [get_url(s) for s in euro_data if get_url(s)]
    euro_urls.extend(euro_url)
    
    url = data["next"]
    time.sleep(0.2)   

print("Total European GRCh38 URLs:", len(euro_urls))

Total European GRCh38 URLs: 2334
